In [1]:
import os

os.environ["HF_HUB_DISABLE_XET"] = "1"

print("Hugging Face Xet downloads disabled.")

Hugging Face Xet downloads disabled.


In [4]:
import itertools
import torch
from datasets import load_dataset
from jiwer import cer, wer
from transformers import (
    TrOCRProcessor,
    VisionEncoderDecoderModel,
    Qwen2VLForConditionalGeneration,
    AutoProcessor,
)
from qwen_vl_utils import process_vision_info

In [5]:
STEP = 50
N_LINES = 300

dataset = load_dataset("CATMuS/medieval", split="test", streaming=True)
sampled_rows = list(
    itertools.islice(
        (row for i, row in enumerate(dataset) if i % STEP == 0),
        N_LINES,
    )
)

print(f"Loaded {len(sampled_rows)} test lines")
print("Languages:", sorted({row["language"] for row in sampled_rows}))

device = "cuda" if torch.cuda.is_available() else "cpu"
print("Device:", device)

README.md:   0%|          | 0.00/7.10k [00:00<?, ?B/s]

Resolving data files:   0%|          | 0/252 [00:00<?, ?it/s]

Resolving data files:   0%|          | 0/43 [00:00<?, ?it/s]

Resolving data files:   0%|          | 0/51 [00:00<?, ?it/s]

Loaded 300 test lines
Languages: ['Castilian', 'Catalan', 'French', 'Italian', 'Latin']
Device: cuda


In [ ]:
trocr_id = "medieval-data/trocr-medieval-base"
trocr_processor = TrOCRProcessor.from_pretrained(trocr_id)
trocr_model = VisionEncoderDecoderModel.from_pretrained(trocr_id).to(device).eval()

In [7]:
def predict_trocr(image):
    inputs = trocr_processor(
        images=image.convert("RGB"),
        return_tensors="pt",
    ).pixel_values.to(device)

    with torch.inference_mode():
        output_ids = trocr_model.generate(
            inputs, max_new_tokens=128, num_beams=4, no_repeat_ngram_size=4
        )

    return trocr_processor.batch_decode(
        output_ids,
        skip_special_tokens=True,
    )[0]

In [8]:
qwen_id = "medieval-data/qwen2-vl-2b-catmus"
qwen_model = Qwen2VLForConditionalGeneration.from_pretrained(
    qwen_id,
    torch_dtype="auto",
    device_map="auto",
).eval()

config.json:   0%|          | 0.00/1.12k [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/4.42G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/729 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/220 [00:00<?, ?B/s]

In [9]:
qwen_processor = AutoProcessor.from_pretrained("Qwen/Qwen2-VL-2B-Instruct")


def predict_qwen(image):
    messages = [
        {
            "role": "user",
            "content": [
                {"type": "image", "image": image.convert("RGB")},
                {"type": "text", "text": "Convert this image to text."},
            ],
        }
    ]

    prompt = qwen_processor.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True,
    )
    image_inputs, video_inputs = process_vision_info(messages)

    inputs = qwen_processor(
        text=[prompt],
        images=image_inputs,
        videos=video_inputs,
        padding=True,
        return_tensors="pt",
    ).to(qwen_model.device)

    with torch.inference_mode():
        output_ids = qwen_model.generate(
            **inputs,
            max_new_tokens=256,
        )

    # Remove the prompt tokens so only the generated transcription remains.
    generated_ids = [
        output[len(input_ids) :]
        for input_ids, output in zip(inputs.input_ids, output_ids)
    ]

    return qwen_processor.batch_decode(
        generated_ids,
        skip_special_tokens=True,
        clean_up_tokenization_spaces=False,
    )[0]

preprocessor_config.json:   0%|          | 0.00/347 [00:00<?, ?B/s]

chat_template.json:   0%|          | 0.00/1.05k [00:00<?, ?B/s]

config.json:   0%|          | 0.00/1.20k [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/4.19k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/2.78M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/1.67M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/7.03M [00:00<?, ?B/s]

In [10]:
print(type(sampled_rows))
print("Number of samples:", len(sampled_rows))

print("\nFirst sample:")
print(sampled_rows[0])

print("\nAvailable columns:")
if hasattr(sampled_rows, "column_names"):
    print(sampled_rows.column_names)
elif hasattr(sampled_rows, "columns"):
    print(sampled_rows.columns)

<class 'list'>
Number of samples: 300

First sample:
{'text': 'fazer merçed ⁊ qnt̃o biẽ yo pudiere Et', 'im': <PIL.PngImagePlugin.PngImageFile image mode=RGB size=1516x187 at 0x7D690896EBA0>, 'language': 'Castilian', 'century': 14, 'region': 'MainZone', 'script_type': 'Hybrida', 'shelfmark': 'Paris, BnF, esp. 36', 'verse': 'prose', 'genre': 'Narratives', 'project': 'Towards General Castilian HTR', 'line_type': 'DefaultLine', 'gen_split': 'test'}

Available columns:


In [11]:
results = []

for index, row in enumerate(sampled_rows, start=1):
    image = row["im"]
    truth = row["text"].strip()

    try:
        trocr_text = predict_trocr(image)
    except Exception as error:
        trocr_text = f"[ERROR: {error}]"

    try:
        qwen_text = predict_qwen(image)
    except Exception as error:
        qwen_text = f"[ERROR: {error}]"

    results.append(
        {
            "language": row.get("language", "unknown"),
            "truth": truth,
            "TrOCR": trocr_text.strip(),
            "Qwen2-VL": qwen_text.strip(),
        }
    )

    if index % 10 == 0:
        print(f"Processed {index}/{len(sampled_rows)}")

[transformers] Ignoring clean_up_tokenization_spaces=True for BPE tokenizer RobertaTokenizer. The clean_up_tokenization post-processing step is designed for WordPiece tokenizers and is destructive for BPE (it strips spaces before punctuation). Set clean_up_tokenization_spaces=False to suppress this warning, or set clean_up_tokenization_spaces_for_bpe_even_though_it_will_corrupt_output=True to force cleanup anyway.


Processed 10/300
Processed 20/300
Processed 30/300
Processed 40/300
Processed 50/300
Processed 60/300
Processed 70/300
Processed 80/300
Processed 90/300
Processed 100/300
Processed 110/300
Processed 120/300
Processed 130/300
Processed 140/300
Processed 150/300
Processed 160/300
Processed 170/300
Processed 180/300
Processed 190/300
Processed 200/300
Processed 210/300
Processed 220/300
Processed 230/300
Processed 240/300
Processed 250/300
Processed 260/300
Processed 270/300
Processed 280/300
Processed 290/300
Processed 300/300


In [12]:
for model_name in ["TrOCR", "Qwen2-VL"]:
    valid_rows = [row for row in results if not row[model_name].startswith("[ERROR:")]

    references = [row["truth"] for row in valid_rows]
    predictions = [row[model_name] for row in valid_rows]

    print(f"\n{model_name}")
    print(f"Successful predictions: {len(valid_rows)}/{len(results)}")
    print(f"CER: {cer(references, predictions):.2%}")
    print(f"WER: {wer(references, predictions):.2%}")


TrOCR
Successful predictions: 300/300
CER: 18.83%
WER: 54.42%

Qwen2-VL
Successful predictions: 300/300
CER: 25.47%
WER: 69.90%


In [13]:
for row in results[:5]:
    print("\n" + "=" * 80)
    print("Language:", row["language"])
    print("Ground truth:", row["truth"])
    print("TrOCR:", row["TrOCR"])
    print("Qwen2-VL:", row["Qwen2-VL"])


Language: Castilian
Ground truth: fazer merçed ⁊ qnt̃o biẽ yo pudiere Et
TrOCR: fazer merçed ⁊ qñto biẽ yo pudiere Et
Qwen2-VL: fazer merced ⁊ q̃nto biẽ yo pudiese Et

Language: Castilian
Ground truth: comigo q̃ero q̃ tomedes por la ynfanta
TrOCR: comigo q̃ero q̃ tomedes por la ynfanta
Qwen2-VL: comigo q̃q̃ tomedes porla infanta

Language: Castilian
Ground truth: ⁊ desant pablo ¶ E otrosi alos que
TrOCR: ⁊ de saut pablo ¶E otrosi alos que
Qwen2-VL: ⁊ desant pablo ¶ E otrosi alos que

Language: Castilian
Ground truth: ça fazia ensus fiiuelos Etq̈nd buenos
TrOCR: ça fazia en sus fiiuelos Etqñd buenes
Qwen2-VL: ca fusia en sus figuelos ꝑtñd buenos

Language: Castilian
Ground truth: entender por esta estoria /¶
TrOCR: entender por esta estoria. ¶
Qwen2-VL: entender por esta estofia
